# NYC Housing × Transportation: Does Affordable Housing Sit Where Serious Crashes Happen?

**Course:** Data Engineering and Visualization — Winter Semester 2026 · **Milestone 1**

**Live website:** `https://dem1-housing-traffic-safety.onrender.com/$0` · **GitHub:** `https://github.com/Hamzawiiz24/dem1-housing-traffic-safety$0`

## Team members and contributions


| Member | Main tasks | Research questions | Charts |
|---|---|---|---|
| **[Hamza Mohamed]** | Dataset selection and loading (Socrata API), column standardisation, EDA (Section 2), housing cleaning (3.1–3.5): comma-number fix, missing values, duplicate buildings, outliers | RQ1, RQ2 | 1, 2 |
| **[Kenzy Ehab]** | SIRTA cleaning (3.6–3.9): junk rows, ID duplicates, domain rules, nearest-neighbour ZIP recovery and validation; ZIP aggregation, outer join, post-integration cleaning (Sections 4–6) | RQ3, RQ4 | 3, 4 |
| **[Abduallah osama]** | Borough-level robustness table, correlation analysis (Section 7), year-by-borough crash-rate analysis, share comparison | RQ5, RQ6 | 5, 8 |
| **[Jana Omara]** | ZIP centroids, spatial analysis and map, housing-group trend analysis over time | RQ7, RQ8 | 6, 7 |
| **[Hazem Shehab]** | Income-mix and construction-type analysis (incl. within-borough tests), Dash website (`app.py`: filters, search parser, Generate Report, story), README and Render deployment | RQ9, RQ10 | 9, 10 |
| **[Eyad Tarek]** | Time-of-day analysis (crash_time parsing, chi-square test), housing-delivery vs crash-trend analysis (completion dates), fatal-crash coverage check | RQ11, RQ12 | 11, 12 |
## Research questions

**Main question:** Do NYC neighbourhoods with the most affordable housing also experience the most serious traffic crashes, and does the answer change by borough, over time, or by the type of housing built?

| # | Research question | Chart |
|---|---|---|
| RQ1 | Do ZIP codes with more affordable units have more serious-injury crashes, and how strong is the relationship? | 1 – scatter |
| RQ2 | Relative to the affordable housing it holds, which borough carries the heaviest serious-crash burden? | 2 – bar |
| RQ3 | Are the ZIPs with the most serious crashes also major affordable-housing locations, or are they housing-poor? | 3 – ranked bar |
| RQ4 | Do the ZIPs receiving the most affordable housing experience above-median crash counts? | 4 – ranked bar + median |
| RQ5 | Has the crash burden per affordable unit changed between 2022 and 2026, and does the trend differ by borough? | 5 – heatmap |
| RQ6 | Is each borough's share of the city's serious crashes proportional to its share of affordable units? | 8 – grouped bar |
| RQ7 | Do serious-crash hotspots and affordable-housing concentrations overlap in space? | 6 – map |
| RQ8 | Do ZIPs with high, medium, low or no affordable housing follow different crash trends over time? | 7 – line |
| RQ9 | Do ZIPs whose affordable housing targets the lowest incomes (extremely-low-income share) have higher crash rates? | 9 – scatter |
| RQ10 | Do ZIPs dominated by new construction have different crash rates than ZIPs dominated by preservation, once borough is accounted for? | 10 – box plot |
| RQ11 | Do serious crashes in high-housing ZIPs happen at different times of day than elsewhere? | 11 – line |
| RQ12 | Did ZIPs that received the most newly completed affordable housing (2022–2025) see serious crashes rise faster? | 12 – bar |

**Why it matters to New Yorkers:** the city builds or preserves thousands of affordable units a year, and Vision Zero targets street-safety investment. If affordable housing sits on dangerous streets, the residents with the fewest alternatives to walking, cycling and transit are the most exposed. Knowing where the two overlap tells planners where safety upgrades would protect the most households.

## 1. Datasets: overview and selection

| | Housing & Development | Transportation |
|---|---|---|
| **Dataset** | Affordable Housing Production by Building | SIRTA — Serious Injury Response, Tracking & Analysis |
| **ID / link** | [`hg8x-zxpr`](https://data.cityofnewyork.us/d/hg8x-zxpr) | [`xeqp-qz8h`](https://data.cityofnewyork.us/d/xeqp-qz8h) |
| **Rows × columns** | 9,551 × 41 | 13,754 × 23 |
| **Unit of record** | One building in one HPD project (Housing New York / Housing Our Neighbors plans) | One crash with ≥ 1 severe injury or fatality |
| **Time coverage** | Project start dates 2014 – Jun 2026 | Crash dates 1 Jan 2022 – 30 Jun 2026 |
| **Geographic fields** | Borough, Postcode, Latitude/Longitude, Community Board, Council District, Census Tract, NTA | Crash Borough, ZipCode, Latitude/Longitude, Community Board, Council District, Census Tract 2020, NTA 2020 |
| **Key measures** | `All Counted Units` (affordable units), units by income band (extremely low … middle), construction type (New Construction / Preservation) | Severe injuries, fatalities, non-severe injuries |
| **Known issues (found in EDA)** | 20.5 % confidential rows with no address/ZIP; `Total Units` stored as text with commas; same building repeated across projects; empty income bands | 444 rows with no crash ID or data; 41 % missing ZIP; placeholder ZIP `10000`; 21 % missing coordinates; one repeated crash ID |

**Why these two datasets.** Both meet the ≥ 9-column requirement and share several geographic keys. SIRTA records only the most serious crashes, which is the safety outcome that matters most for residents, and it is curated by NYPD/DOT, so it is cleaner than the full Motor Vehicle Collisions table (2M+ rows). The housing dataset is the city's official record of where affordable units were financed. *Feasibility check (if SIRTA is not in the course catalog):* 13,754 rows × 23 columns, ZIP + coordinates + borough for integration, 4.5 years of dates; after ZIP recovery 86 % of crashes can be placed in a ZIP, giving 189 integrated ZIPs (enough for correlation tests at citywide and borough level).

**Aggregation plan.** The datasets have different units (buildings vs crashes) and no shared record ID, so we do not match individual records. We aggregate both to **ZIP code** (Section 4): fine enough to separate neighbourhoods (189 ZIPs) yet coarse enough that most ZIPs contain both housing and crashes. We also keep a ZIP × construction-type table and a ZIP × crash-year table so the website can filter by a housing category and a transportation time period.

**Alternatives considered.** *Census tract* (finer, but SIRTA's tract field uses 2020 boundaries while HPD uses 2010, and most tracts would have zero crashes). *Community district* (59 units — too coarse for a correlation). *Point-level distance* (crashes within X m of each building) — more precise, but ZIP keeps the analysis interpretable for the website and is the level the brief suggests.

In [47]:
# Run once in a fresh environment (scipy is needed for Spearman correlations and the Mann-Whitney test)
%pip install pandas numpy scipy plotly dash

In [48]:
# ============================================================
# Imports and global settings
# ============================================================
import re
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from scipy.spatial import cKDTree

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)

TEMPLATE = 'plotly_white'
BOROUGH_ORDER = ['Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island']
BOROUGH_COLORS = {'Bronx': '#1f77b4', 'Brooklyn': '#d62728', 'Manhattan': '#2ca02c',
                  'Queens': '#ff7f0e', 'Staten Island': '#9467bd'}

# NYC bounding box used as a domain rule for coordinates
NYC_LAT = (40.49, 40.92)
NYC_LON = (-74.26, -73.69)

# Minimum affordable units for a ZIP to receive a rate (crashes per 1,000 units).
# Below this, one or two crashes swing the ratio by thousands (see Section 7.4).
MIN_UNITS_FOR_RATE = 100

# Fraction of a year covered by 2026 data (SIRTA ends 30 June 2026) - used to annualise
YEAR_FRACTION = {2022: 1.0, 2023: 1.0, 2024: 1.0, 2025: 1.0, 2026: 181 / 365}

### Loading the data
The cell below downloads both datasets through the NYC Open Data API when `DOWNLOAD = True` (the code from the project brief). We ran it once and saved the files as `housing_raw.csv` and `sirta_raw.csv`, so the notebook reruns offline with exactly the same data. Both datasets are small enough (< 15k rows) to download in full; no sampling was needed.

In [49]:
# ============================================================
# Load the raw data
# ============================================================
# Set DOWNLOAD = True to pull fresh copies from NYC Open Data (Socrata API).
# We saved the downloads as CSV so the notebook is reproducible offline.
DOWNLOAD = False

HOUSING_ID = 'hg8x-zxpr'   # Affordable Housing Production by Building
SIRTA_ID = 'xeqp-qz8h'     # SIRTA - Serious Injury Response, Tracking & Analysis
BASE_URL = 'https://data.cityofnewyork.us/api/views/{}/rows.csv?accessType=DOWNLOAD'

if DOWNLOAD:
    housing_raw = pd.read_csv(BASE_URL.format(HOUSING_ID), low_memory=False)
    sirta_raw = pd.read_csv(BASE_URL.format(SIRTA_ID), low_memory=False)
    housing_raw.to_csv('housing_raw.csv', index=False)
    sirta_raw.to_csv('sirta_raw.csv', index=False)
else:
    housing_raw = pd.read_csv('housing_raw.csv', low_memory=False)
    sirta_raw = pd.read_csv('sirta_raw.csv', low_memory=False)

print('Housing raw:', housing_raw.shape)
print('SIRTA raw:  ', sirta_raw.shape)

Housing raw: (9551, 41)
SIRTA raw:   (13754, 23)


Column names differ between the API and the CSV export and contain spaces, parentheses and BOM characters, so we normalise them to `snake_case` first.

In [50]:
# ============================================================
# Standardise column names: 'Building ID' -> 'building_id'
# ============================================================
def normalize_col(col):
    """Lower-case, strip BOM/parentheses, replace spaces/dashes with '_'."""
    col = col.strip().lstrip('﻿').lower()
    col = re.sub(r'[\(\)]', '', col)
    col = re.sub(r'[\s\-&]+', '_', col)
    col = re.sub(r'_+', '_', col).strip('_')
    if re.match(r'^\d', col):          # '1_br_units' -> '_1_br_units'
        col = '_' + col
    return col

housing = housing_raw.rename(columns=normalize_col)
sirta = sirta_raw.rename(columns=normalize_col)

print('Housing columns:', housing.columns.tolist(), '\n')
print('SIRTA columns:  ', sirta.columns.tolist())

Housing columns: ['project_id', 'project_name', 'project_start_date', 'project_completion_date', 'building_id', 'number', 'street', 'borough', 'postcode', 'bbl', 'bin', 'community_board', 'council_district', 'census_tract', 'nta_neighborhood_tabulation_area', 'latitude', 'longitude', 'latitude_internal', 'longitude_internal', 'building_completion_date', 'reporting_construction_type', 'extended_affordability_only', 'prevailing_wage_status', 'extremely_low_income_units', 'very_low_income_units', 'low_income_units', 'moderate_income_units', 'middle_income_units', 'other_income_units', 'studio_units', '_1_br_units', '_2_br_units', '_3_br_units', '_4_br_units', '_5_br_units', '_6_br+_units', 'unknown_br_units', 'counted_rental_units', 'counted_homeownership_units', 'all_counted_units', 'total_units'] 

SIRTA columns:   ['collision_id', 'crash_date', 'crash_time', 'crash_police_precinct', 'crash_borough', 'total_non_severe_injuries', 'total_severe_injuries', 'total_fatalities', 'latitude', '

## 2. Exploratory data analysis (EDA)
### 2.1 Structure, types and missing values

In [51]:
# ============================================================
# 2.1 Structure overview: dtypes, missingness, uniqueness
# ============================================================
def overview(df, cols):
    return pd.DataFrame({
        'dtype': df[cols].dtypes.astype(str),
        'missing': df[cols].isna().sum(),
        'missing_%': (df[cols].isna().mean() * 100).round(1),
        'unique': df[cols].nunique(),
    })

housing_key_cols = ['project_id', 'building_id', 'borough', 'postcode', 'latitude', 'longitude',
                    'project_start_date', 'reporting_construction_type',
                    'extremely_low_income_units', 'very_low_income_units', 'low_income_units',
                    'moderate_income_units', 'middle_income_units', 'other_income_units',
                    'all_counted_units', 'total_units']
sirta_key_cols = ['collision_id', 'crash_date', 'crash_time', 'crash_borough', 'zipcode',
                  'latitude', 'longitude', 'total_severe_injuries', 'total_fatalities',
                  'total_non_severe_injuries']

print('=== Housing ===')
print(overview(housing, housing_key_cols))
print('\n=== SIRTA ===')
print(overview(sirta, sirta_key_cols))

=== Housing ===
                               dtype  missing  missing_%  unique
project_id                     int64        0        0.0    5928
building_id                  float64     1949       20.4    7478
borough                       object        0        0.0       5
postcode                     float64     1958       20.5     158
latitude                     float64     1958       20.5    7246
longitude                    float64     1958       20.5    7183
project_start_date            object        0        0.0    2346
reporting_construction_type   object        0        0.0       2
extremely_low_income_units   float64     7470       78.2     190
very_low_income_units        float64     6151       64.4     208
low_income_units             float64     4046       42.4     228
moderate_income_units        float64     7857       82.3     104
middle_income_units          float64     6860       71.8     153
other_income_units           float64     8191       85.8       5
all_count

In [52]:
# Visual check of missingness in the two join-relevant fields (ZIP and coordinates)
miss = pd.DataFrame({
    'dataset': ['Housing', 'Housing', 'SIRTA', 'SIRTA'],
    'field': ['ZIP code', 'Latitude/Longitude', 'ZIP code', 'Latitude/Longitude'],
    'missing_%': [housing['postcode'].isna().mean() * 100, housing['latitude'].isna().mean() * 100,
                  sirta['zipcode'].isna().mean() * 100, sirta['latitude'].isna().mean() * 100],
})
fig = px.bar(miss, x='field', y='missing_%', color='dataset', barmode='group', text_auto='.1f',
             title='EDA - Share of records missing the geographic join fields',
             labels={'missing_%': 'Records missing (%)', 'field': 'Field', 'dataset': 'Dataset'})
fig.update_layout(template=TEMPLATE, height=420)
fig.show()

**What the overview shows**
- **Housing:** `postcode`, `latitude`, `longitude` and `building_id` are missing together for ~1,950 rows (the confidential homeownership buildings). Income-band columns are 42–86 % empty. `total_units` is text, not a number. `all_counted_units` is missing for 29 rows.
- **SIRTA:** exactly 444 rows are empty in every crash field (date, injuries, ID). `zipcode` is missing for 41 % of rows and coordinates for 21 %, so ZIP missingness is the main threat to the integration.

### 2.2 Descriptive statistics and distributions

In [53]:
# ============================================================
# 2.2 Descriptive statistics and distributions
# ============================================================
# 'total_units' is stored as text in the raw file - find out why before converting
tu_numeric = pd.to_numeric(housing['total_units'], errors='coerce')
bad_tu = housing.loc[tu_numeric.isna() & housing['total_units'].notna(), 'total_units']
print('total_units values that are not plain numbers:', bad_tu.tolist())

print('\nHousing - unit counts per building:')
print(housing[['all_counted_units', 'extremely_low_income_units',
               'very_low_income_units', 'low_income_units']].describe().round(1))
print('\nSIRTA - injuries per crash:')
print(sirta[['total_severe_injuries', 'total_fatalities',
             'total_non_severe_injuries']].describe().round(2))

print('\nHousing rows by borough:\n', housing['borough'].value_counts(dropna=False))
print('\nSIRTA rows by borough:\n', sirta['crash_borough'].value_counts(dropna=False))
print('\nHousing construction type:\n', housing['reporting_construction_type'].value_counts())

total_units values that are not plain numbers: ['1,320', '1,046', '1,122', '1,000', '1,175', '1,028', '1,175']

Housing - unit counts per building:
       all_counted_units  extremely_low_income_units  very_low_income_units  low_income_units
count             9522.0                      2081.0                 3400.0            5505.0
mean                33.7                        28.3                   25.5              19.8
std                 65.8                        44.6                   49.3              40.7
min                  1.0                         1.0                    1.0               1.0
25%                  2.0                         2.0                    3.0               1.0
50%                  8.0                         8.0                    7.0               5.0
75%                 30.0                        37.0                   21.0              19.0
max                917.0                       429.0                  453.0             536.0

SIRTA

**Observations:** `total_units` is text because seven large buildings are written with a thousands separator (`'1,320'`). A plain `pd.to_numeric(errors='coerce')` would silently turn them into NaN and lose 7,866 units, so we strip commas in 3.1. Unit counts per building are extremely right-skewed (median 8, max 917), which is why we use log axes and rank-based (Spearman) statistics later. SIRTA crashes almost always have exactly one severe injury (median 1, max 7).

In [54]:
fig = px.histogram(housing, x='all_counted_units', nbins=80, log_y=True,
                   title='EDA - Affordable (counted) units per building (log y-axis)',
                   labels={'all_counted_units': 'Affordable units in building'})
fig.update_layout(template=TEMPLATE, height=420, yaxis_title='Number of buildings (log)')
fig.show()

The histogram confirms a long right tail: most buildings are small, a few hundred are large developments.

In [55]:
crash_year = pd.to_datetime(sirta['crash_date'], format='%m/%d/%Y', errors='coerce').dt.year
proj_year = pd.to_datetime(housing['project_start_date'], format='%m/%d/%Y', errors='coerce').dt.year
by_year = pd.concat([
    crash_year.value_counts().rename('Serious-injury crashes (SIRTA)'),
    proj_year.value_counts().rename('Affordable-housing buildings started (HPD)'),
], axis=1).sort_index()
by_year.index = by_year.index.astype(int)
by_year.index.name = 'year'
fig = px.line(by_year.reset_index().melt(id_vars='year', var_name='series', value_name='records'),
              x='year', y='records', color='series', markers=True,
              title='EDA - Records per year in each dataset (2026 = Jan-Jun only)',
              labels={'year': 'Year', 'records': 'Records', 'series': ''})
fig.update_layout(template=TEMPLATE, height=420)
fig.show()
print(by_year)

      Serious-injury crashes (SIRTA)  Affordable-housing buildings started (HPD)
year                                                                            
2014                             NaN                                         734
2015                             NaN                                         694
2016                             NaN                                         722
2017                             NaN                                         703
2018                             NaN                                         691
2019                             NaN                                         816
2020                             NaN                                         608
2021                             NaN                                         780
2022                          2879.0                                         670
2023                          3016.0                                         872
2024                        

**Temporal coverage:** crashes run from 2022 to June 2026 (≈ 2,900–3,100 per full year, 1,352 in the first half of 2026), while housing starts go back to 2014. The time windows only partly overlap. Most housing was in place before the crashes we observe, but we cannot say whether any crash happened before or after a nearby building opened. For year comparisons we annualise 2026 (× 365/181).

## 3. Pre-integration cleaning
### 3.1 Housing — type conversion

In [56]:
# ============================================================
# 3.1 HOUSING - type conversion
# ============================================================
h = housing.copy()

# 'total_units' contains values such as '1,320'. pd.to_numeric(errors='coerce') would
# silently turn these into NaN and drop 7 of the largest buildings, so strip commas first.
h['total_units'] = pd.to_numeric(h['total_units'].astype(str).str.replace(',', '', regex=False),
                                 errors='coerce')

band_cols = ['extremely_low_income_units', 'very_low_income_units', 'low_income_units',
             'moderate_income_units', 'middle_income_units', 'other_income_units']
num_cols = band_cols + ['all_counted_units', 'latitude', 'longitude', 'building_id']
for c in num_cols:
    h[c] = pd.to_numeric(h[c], errors='coerce')

print('total_units still non-numeric after fix:', h['total_units'].isna().sum())
print('Units recovered by the comma fix:', int(h.loc[bad_tu.index, 'total_units'].sum()))

total_units still non-numeric after fix: 0
Units recovered by the comma fix: 7866


### 3.2 Housing — missing values (drop vs impute)
| Field | Missing | Decision | Justification |
|---|---|---|---|
| Income bands | 42–86 % | **Impute 0** | HPD leaves a band blank when a building has no units in it. Verified: with blanks = 0, the bands add up exactly to `all_counted_units` in 100 % of rows. |
| `all_counted_units` | 29 rows | **Impute 0** | All income bands are also blank for these rows, so the building reports no affordable units. |
| `postcode` / coordinates | 1,958 rows | **Drop from ZIP analysis** | 1,949 are `CONFIDENTIAL` (privacy-protected homeownership units) with no address. There is nothing to impute from, and they hold only 1.0 % of affordable units. |

In [57]:
# ============================================================
# 3.2 HOUSING - missing values
# ============================================================
# (a) Income bands: HPD leaves a band empty when the building has no units in it.
#     Check: bands (NaN -> 0) must add up exactly to all_counted_units.
band_sum = h[band_cols].fillna(0).sum(axis=1)
identity = (band_sum == h['all_counted_units'].fillna(0)).mean()
print(f'Rows where income bands (NaN->0) sum to all_counted_units: {identity:.1%}')
h[band_cols] = h[band_cols].fillna(0)

# (b) all_counted_units missing (29 rows): bands are all zero for these rows, so the
#     building reports no affordable units -> impute 0 (consistent with the identity above).
print('all_counted_units missing:', h['all_counted_units'].isna().sum())
h['all_counted_units'] = h['all_counted_units'].fillna(0)

# (c) Confidential buildings: no building ID, address, ZIP or coordinates.
conf = h['postcode'].isna()
print(f"Rows without ZIP: {conf.sum()} "
      f"({h.loc[conf, 'project_name'].eq('CONFIDENTIAL').sum()} are labelled CONFIDENTIAL)")
print(f"Share of affordable units in those rows: "
      f"{h.loc[conf, 'all_counted_units'].sum() / h['all_counted_units'].sum():.1%}")
# They cannot be placed in a ZIP, so they are excluded from the ZIP analysis (no imputation possible).
h = h[~conf].copy()
print('Housing rows kept:', len(h))

Rows where income bands (NaN->0) sum to all_counted_units: 100.0%
all_counted_units missing: 29
Rows without ZIP: 1958 (1949 are labelled CONFIDENTIAL)
Share of affordable units in those rows: 1.0%
Housing rows kept: 7593


### 3.3 Housing — standardise formats
ZIP becomes a 5-character string (so `10001` and `10001.0` match the crash data), borough and construction type are title-cased, and the date columns are parsed with an explicit format.

In [58]:
# ============================================================
# 3.3 HOUSING - standardise formats
# ============================================================
h['postcode'] = h['postcode'].astype(int).astype(str).str.zfill(5)
h['borough'] = h['borough'].str.strip().str.title()
h['construction_type'] = h['reporting_construction_type'].str.strip().str.title()
for c in ['project_start_date', 'project_completion_date', 'building_completion_date']:
    h[c] = pd.to_datetime(h[c], format='%m/%d/%Y', errors='coerce')
h['project_start_year'] = h['project_start_date'].dt.year

print('ZIPs valid (10xxx/11xxx):', h['postcode'].str.match(r'^1[01]\d{3}$').all())
print('Boroughs:', sorted(h['borough'].unique()))
print('Construction types:', h['construction_type'].value_counts().to_dict())

ZIPs valid (10xxx/11xxx): True
Boroughs: ['Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island']
Construction types: {'Preservation': 4038, 'New Construction': 3555}


### 3.4 Housing — duplicates
There are no fully duplicated rows, but 245 rows share a `building_id` with another row (122 buildings). They are the same building financed in two projects, e.g. preserved in 2019 and again in 2023. We measure the *stock* of affordable housing, so we keep one row per building (the most recent project). This removes 10,130 units that would otherwise be counted twice.

In [59]:
# ============================================================
# 3.4 HOUSING - duplicates
# ============================================================
print('Fully duplicated rows:', h.duplicated().sum())
dup_mask = h['building_id'].duplicated(keep=False)
print('Rows sharing a building_id with another row:', dup_mask.sum(),
      f"({h.loc[dup_mask, 'building_id'].nunique()} buildings)")
print(h.loc[dup_mask].sort_values('building_id')[
    ['project_id', 'building_id', 'project_start_date', 'construction_type', 'all_counted_units']].head(6))

# The same physical building appears once per HPD project (e.g. preserved in 2019 and again
# in 2023). Summing all rows would count the same apartments twice. Our question is about the
# housing stock, so we keep ONE row per building: the most recent project.
before_units = h['all_counted_units'].sum()
h = (h.sort_values('project_start_date')
      .drop_duplicates(subset='building_id', keep='last')
      .reset_index(drop=True))
print(f'Buildings after dedup: {len(h)}; double-counted units removed: '
      f'{int(before_units - h["all_counted_units"].sum()):,}')

Fully duplicated rows: 0
Rows sharing a building_id with another row: 245 (122 buildings)
      project_id  building_id project_start_date construction_type  all_counted_units
5521       65392       1338.0         2019-06-27      Preservation              208.0
2843       70493       1338.0         2023-06-23      Preservation              208.0
2844       70493       1339.0         2023-06-23      Preservation               92.0
5522       65392       1339.0         2019-06-27      Preservation               92.0
5523       65392       2098.0         2019-06-27      Preservation              208.0
2845       70493       2098.0         2023-06-23      Preservation              208.0
Buildings after dedup: 7470; double-counted units removed: 10,130


### 3.5 Housing — outliers and domain rules
Domain rules (counted ≤ total units, coordinates inside NYC, no negative counts) find **no violations**. The IQR rule flags 910 buildings above 98 units (12 %), but these are real large developments (e.g. 917 units at 711 Seagirt, Far Rockaway), not errors. **Decision: keep them.** Removing them would delete most of the city's affordable housing. We handle the skew with log axes and rank-based statistics instead.

In [60]:
# ============================================================
# 3.5 HOUSING - outliers and domain rules
# ============================================================
# Domain rules
print('Counted units > total units:', (h['all_counted_units'] > h['total_units']).sum())
print('Coordinates outside NYC box:',
      (~h['latitude'].between(*NYC_LAT) | ~h['longitude'].between(*NYC_LON)).sum())
print('Negative unit counts:', (h[band_cols + ['all_counted_units']] < 0).any(axis=1).sum())

# IQR rule on affordable units per building
q1, q3 = h['all_counted_units'].quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)
h['units_outlier'] = h['all_counted_units'] > upper
print(f'\nIQR upper fence: {upper:.0f} units; buildings above it: {h["units_outlier"].sum()} '
      f'({h["units_outlier"].mean():.1%})')
print(h.nlargest(5, 'all_counted_units')[['project_name', 'postcode', 'borough', 'all_counted_units']])

Counted units > total units: 0
Coordinates outside NYC box: 0
Negative unit counts: 0

IQR upper fence: 98 units; buildings above it: 910 (12.2%)
                      project_name postcode    borough  all_counted_units
1990             711 SEA GIRT HDFC    11691     Queens              917.0
5015                    NORTH COVE    10034  Manhattan              611.0
4754                 3333 BROADWAY    10031  Manhattan              565.0
3829                 BRONX POINT A    10451      Bronx              542.0
1438  JAMAICA CROSSING (HIGH-RISE)    11435     Queens              539.0


In [61]:
fig = px.box(h, x='borough', y='all_counted_units', color='borough', points='outliers',
             category_orders={'borough': BOROUGH_ORDER}, color_discrete_map=BOROUGH_COLORS,
             title='Cleaning check - affordable units per building, IQR outliers shown as points',
             labels={'all_counted_units': 'Affordable units per building', 'borough': 'Borough'})
fig.update_layout(template=TEMPLATE, height=450, showlegend=False)
fig.show()

### 3.6 SIRTA — junk rows and duplicates
The 444 rows with no `collision_id` contain no crash at all, only geography columns, so they are dropped. (The original notebook mis-reported these as "444 duplicate IDs".) One real ID, 4597666, appears twice. The two rows are crashes three years apart (13 Jan 2023 and 13 Jan 2026) with different outcomes, which suggests an ID entry error, not a duplicate. We keep both and count **rows**, not unique IDs.

In [62]:
# ============================================================
# 3.6 SIRTA - junk rows and duplicates
# ============================================================
s = sirta.copy()
junk = s['collision_id'].isna()
print('Rows without collision_id:', junk.sum())
print('...of which have a crash date / injury data:',
      s.loc[junk, ['crash_date', 'total_severe_injuries']].notna().any(axis=1).sum())
# These rows only carry geography columns (community board, census tract...) - no crash.
s = s[~junk].copy()

print('Fully duplicated rows after removing junk:', s.duplicated().sum())
dup_ids = s[s['collision_id'].duplicated(keep=False)]
print('Repeated collision_id:\n', dup_ids[['collision_id', 'crash_date', 'crash_time',
                                           'total_fatalities', 'on_street']])
# The repeated ID belongs to two crashes three years apart with different streets and
# outcomes -> two distinct events with an ID entry error. We keep both and count ROWS,
# not unique IDs, from here on.
s['crash_key'] = np.arange(len(s))
print('SIRTA rows kept:', len(s))

Rows without collision_id: 444
...of which have a crash date / injury data: 0
Fully duplicated rows after removing junk: 0
Repeated collision_id:
        collision_id  crash_date crash_time  total_fatalities        on_street
2157      4597666.0  01/13/2023      21:01               0.0  WEST 170 STREET
10709     4597666.0  01/13/2026      21:01               1.0  WEST 170 STREET
SIRTA rows kept: 13310


### 3.7 SIRTA — types, formats, domain rules and outliers

In [63]:
# ============================================================
# 3.7 SIRTA - types, formats, domain rules
# ============================================================
inj_cols = ['total_severe_injuries', 'total_fatalities', 'total_non_severe_injuries']
for c in inj_cols + ['latitude', 'longitude', 'zipcode']:
    s[c] = pd.to_numeric(s[c], errors='coerce')
print('Missing injury values:', s[inj_cols].isna().sum().to_dict())

s['crash_date'] = pd.to_datetime(s['crash_date'], format='%m/%d/%Y', errors='coerce')
s['year'] = s['crash_date'].dt.year
s['crash_borough'] = s['crash_borough'].str.strip().str.title()
s['total_injured'] = s['total_severe_injuries'] + s['total_non_severe_injuries']

print('Unparseable dates:', s['crash_date'].isna().sum(),
      '| range:', s['crash_date'].min().date(), '->', s['crash_date'].max().date())
# Domain rule: SIRTA only contains crashes with >= 1 severe injury or fatality
no_serious = (s['total_severe_injuries'] + s['total_fatalities']) == 0
print('Crashes with no severe injury and no fatality:', no_serious.sum())
print('Negative counts:', (s[inj_cols] < 0).any(axis=1).sum())
print('Coordinates outside NYC box:',
      (s['latitude'].notna() & (~s['latitude'].between(*NYC_LAT) | ~s['longitude'].between(*NYC_LON))).sum())

# Outliers in people hurt per crash.
# IQR first: Q1 = Q3 = 1, so the fence is 1 and every crash with 2+ injured would be an
# "outlier" - the IQR rule degenerates on this distribution.
q1, q3 = s['total_injured'].quantile([0.25, 0.75])
print(f'IQR: Q1={q1}, Q3={q3} -> fence {q3 + 1.5 * (q3 - q1)} (would flag '
      f'{(s["total_injured"] > q3 + 1.5 * (q3 - q1)).sum()} crashes - not meaningful)')
# Domain rule instead: 5+ people hurt = multi-casualty crash
s['injury_outlier'] = s['total_injured'] >= 5
print('Distribution of people hurt per crash:', s['total_injured'].value_counts().sort_index().to_dict())
print(f'Multi-casualty crashes (5+ hurt): {s["injury_outlier"].sum()}; max = {s["total_injured"].max():.0f}')
# These are real events (e.g. a bus crash), not errors -> kept, but we use crash COUNTS as
# the main metric so a single multi-casualty crash cannot dominate a ZIP.

Missing injury values: {'total_severe_injuries': 0, 'total_fatalities': 0, 'total_non_severe_injuries': 0}
Unparseable dates: 0 | range: 2022-01-01 -> 2026-06-30
Crashes with no severe injury and no fatality: 0
Negative counts: 0
Coordinates outside NYC box: 0
IQR: Q1=1.0, Q3=1.0 -> fence 1.0 (would flag 2680 crashes - not meaningful)
Distribution of people hurt per crash: {0.0: 748, 1.0: 9882, 2.0: 1724, 3.0: 552, 4.0: 207, 5.0: 100, 6.0: 47, 7.0: 17, 8.0: 15, 9.0: 3, 10.0: 3, 11.0: 3, 12.0: 2, 13.0: 1, 15.0: 3, 17.0: 1, 18.0: 1, 21.0: 1}
Multi-casualty crashes (5+ hurt): 197; max = 21


**Results:** no missing injury values remain after removing junk rows, every date parses, every crash has ≥ 1 severe injury or fatality (consistent with SIRTA's definition), and no coordinates fall outside NYC. The IQR rule is useless here (Q1 = Q3 = 1), so we use a **domain rule** and flag crashes with 5+ people hurt (197 multi-casualty crashes, up to 21 people). They are real events and are kept. Our main metric is the crash *count*, which a single bus crash cannot inflate.

### 3.8 SIRTA — ZIP code: placeholder and missing values

In [64]:
# ============================================================
# 3.8 SIRTA - ZIP code: placeholder and missing values
# ============================================================
s['zipcode'] = s['zipcode'].astype('Int64').astype(str).replace('<NA>', pd.NA)
placeholder = s['zipcode'].eq('10000')
print('Placeholder ZIP 10000 (not a real NYC ZIP):', placeholder.sum())
s.loc[placeholder, 'zipcode'] = pd.NA

missing_zip = s['zipcode'].isna()
print(f'Crashes without a ZIP: {missing_zip.sum()} ({missing_zip.mean():.1%})')
print('...of which have coordinates:', (missing_zip & s['latitude'].notna()).sum())
print('Missing-ZIP share by borough:\n',
      s.groupby('crash_borough')['zipcode'].apply(lambda z: round(z.isna().mean() * 100, 1)))

Placeholder ZIP 10000 (not a real NYC ZIP): 44
Crashes without a ZIP: 5282 (39.7%)
...of which have coordinates: 4097
Missing-ZIP share by borough:
 crash_borough
Bronx            41.8
Brooklyn         34.2
Manhattan        37.7
Queens           41.9
Staten Island    55.8
Name: zipcode, dtype: float64


**Problem:** 40 % of crashes have no usable ZIP (missing, or the placeholder `10000`), and the share varies by borough (34 % in Brooklyn, 56 % in Staten Island). Simply dropping them, as the first version did, under-counts crashes unevenly across boroughs. That biases exactly the borough comparison we want to make.

### 3.9 SIRTA — recover missing ZIPs (nearest-neighbour imputation)
4,097 of the ZIP-less crashes have coordinates. We assign each one the ZIP of the **nearest point with a known ZIP** (any SIRTA crash with a reported ZIP or any affordable-housing building), using a KD-tree on projected coordinates. Before using the method we **validate** it: we hide the reported ZIP of every labelled crash and predict it from its nearest other point.

In [65]:
# ============================================================
# 3.9 SIRTA - recover missing ZIPs with nearest-neighbour matching
# ============================================================
# Dropping 38% of crashes would bias the crash counts. Most of these crashes have
# coordinates, so we give each one the ZIP of the nearest point whose ZIP IS known
# (other SIRTA crashes + affordable-housing buildings). We validate the method first.

def to_xy(lat, lon):
    """Approximate metres east/north (equirectangular projection around NYC)."""
    return np.column_stack([lon * 111_320 * np.cos(np.radians(40.7)), lat * 110_574])

ref = pd.concat([
    s.loc[s['zipcode'].notna() & s['latitude'].notna(), ['latitude', 'longitude', 'zipcode']],
    h.loc[h['latitude'].notna(), ['latitude', 'longitude']].assign(zipcode=h['postcode']),
], ignore_index=True)
ref_xy = to_xy(ref['latitude'].values, ref['longitude'].values)
tree = cKDTree(ref_xy)

# Validation: hide the ZIP of each labelled crash and predict it from its nearest *other* point
lab = s[s['zipcode'].notna() & s['latitude'].notna()]
dist, idx = tree.query(to_xy(lab['latitude'].values, lab['longitude'].values), k=2)
pred = ref['zipcode'].values[idx[:, 1]]          # k=2 -> skip the point itself
pred_dist = dist[:, 1]
acc = (pred == lab['zipcode'].values)
for cap in [100, 250, 500, 1000]:
    m = pred_dist <= cap
    print(f'Validation - nearest point within {cap:>4} m: accuracy {acc[m].mean():.1%} '
          f'(covers {m.mean():.1%} of crashes)')

Validation - nearest point within  100 m: accuracy 95.7% (covers 68.6% of crashes)
Validation - nearest point within  250 m: accuracy 93.8% (covers 93.3% of crashes)
Validation - nearest point within  500 m: accuracy 93.3% (covers 98.9% of crashes)
Validation - nearest point within 1000 m: accuracy 93.1% (covers 99.9% of crashes)


**Validation result:** when the nearest known point is within 250 m, the predicted ZIP is correct **93.8 %** of the time, and this distance covers 93 % of crashes. Errors come from crashes right on a ZIP boundary, where either neighbouring ZIP is plausible. We therefore accept a match only within **250 m**. (Alternative considered: a spatial join to official ZIP polygons. More exact, but it needs a shapefile download and geopandas; the validated nearest-neighbour approach is accurate enough at ZIP level.)

In [66]:
MAX_DIST_M = 250   # chosen from the validation above: high accuracy, high coverage

to_fill = s['zipcode'].isna() & s['latitude'].notna()
dist, idx = tree.query(to_xy(s.loc[to_fill, 'latitude'].values, s.loc[to_fill, 'longitude'].values), k=1)
cand = pd.Series(ref['zipcode'].values[idx], index=s.index[to_fill])
ok = pd.Series(dist <= MAX_DIST_M, index=cand.index)
s['zip_source'] = np.where(s['zipcode'].notna(), 'reported', 'missing')
s.loc[ok[ok].index, 'zipcode'] = cand[ok]
s.loc[ok[ok].index, 'zip_source'] = 'nearest-neighbour'

print(s['zip_source'].value_counts())
print(f"Crashes with a ZIP: {s['zipcode'].notna().mean():.1%} (was {1 - missing_zip.mean():.1%})")

zip_source
reported             8028
nearest-neighbour    3391
missing              1891
Name: count, dtype: int64
Crashes with a ZIP: 85.8% (was 60.3%)


**Outcome:** 3,391 crashes recover a ZIP, so coverage rises from 60.3 % to **85.8 %**. 1,891 crashes (14 %) still cannot be placed (no coordinates, or > 250 m from any known point) and are excluded from the ZIP analysis. This remains a limitation (Section 9).

In [67]:
fig = px.bar(s.groupby(['crash_borough', 'zip_source']).size().reset_index(name='crashes'),
             x='crash_borough', y='crashes', color='zip_source',
             category_orders={'crash_borough': BOROUGH_ORDER,
                              'zip_source': ['reported', 'nearest-neighbour', 'missing']},
             title='Cleaning check - where each crash ZIP comes from, by borough',
             labels={'crash_borough': 'Borough', 'crashes': 'Crashes', 'zip_source': 'ZIP source'})
fig.update_layout(template=TEMPLATE, height=430)
fig.show()

In [68]:
# Crashes that still have no ZIP (no coordinates, or > 250 m from any known point)
s_zip = s[s['zipcode'].notna()].copy()
print('Crashes excluded from the ZIP analysis:', len(s) - len(s_zip))

Crashes excluded from the ZIP analysis: 1891


## 4. Aggregation to ZIP level
- **Housing → ZIP × construction type** (`housing_zip_type`) and **ZIP** (`housing_zip`): buildings, total units, affordable (counted) units, units by income band, share of units that are new construction.
- **SIRTA → ZIP × year** (`crashes_zip_year`) and **ZIP** (`crashes_zip`): crashes (row count), severe injuries, fatalities, people injured.

**Important correction:** we use `all_counted_units` as *affordable units*. `total_units` includes market-rate apartments in mixed-income buildings (e.g. ZIP 11101: 13,462 total vs 4,739 affordable), so the first version over-stated affordable housing in mixed-income areas.

In [69]:
# ============================================================
# 4. AGGREGATION to ZIP level
# ============================================================
# Housing: one row per ZIP x construction type (keeps the housing category for the app filter)
housing_zip_type = (h.groupby(['postcode', 'construction_type'])
                     .agg(buildings=('building_id', 'nunique'),
                          total_units=('total_units', 'sum'),
                          counted_units=('all_counted_units', 'sum'),
                          extremely_low_income_units=('extremely_low_income_units', 'sum'),
                          very_low_income_units=('very_low_income_units', 'sum'),
                          low_income_units=('low_income_units', 'sum'))
                     .reset_index())

# Housing: one row per ZIP
housing_zip = (housing_zip_type.drop(columns='construction_type')
               .groupby('postcode').sum().reset_index())
new_units = (housing_zip_type[housing_zip_type['construction_type'] == 'New Construction']
             .set_index('postcode')['counted_units'])
housing_zip['new_construction_share'] = (housing_zip['postcode'].map(new_units).fillna(0)
                                         / housing_zip['counted_units'].replace(0, np.nan))

# Crashes: one row per ZIP x year (keeps the transport time dimension for the app filter)
crashes_zip_year = (s_zip.groupby(['zipcode', 'year'])
                    .agg(crashes=('crash_key', 'count'),
                         severe_injuries=('total_severe_injuries', 'sum'),
                         fatalities=('total_fatalities', 'sum'),
                         total_injured=('total_injured', 'sum'))
                    .reset_index())
crashes_zip = (crashes_zip_year.drop(columns='year').groupby('zipcode').sum().reset_index())

print('housing_zip_type:', housing_zip_type.shape, '| housing_zip:', housing_zip.shape)
print('crashes_zip_year:', crashes_zip_year.shape, '| crashes_zip:', crashes_zip.shape)

housing_zip_type: (282, 8) | housing_zip: (158, 8)
crashes_zip_year: (868, 6) | crashes_zip: (187, 5)


## 5. Integration
**Join key: 5-digit ZIP code** (justified in Section 1). Both sides are aggregated first, so the join is one-to-one.

**Join type: full outer join.** 156 ZIPs appear in both datasets, 31 ZIPs have serious crashes but no affordable housing (462 crashes), and 2 ZIPs have housing but no recorded serious crash. An inner join (used in the first version) would drop the 31 no-housing ZIPs. Those ZIPs are exactly the comparison group needed to ask whether housing ZIPs are different. A ZIP's absence from one dataset means *zero records*, because each dataset is a complete register of its events, so the missing counts are true zeros (Section 6.1).

In [70]:
# ============================================================
# 5. INTEGRATION - ZIP-level join
# ============================================================
h_zips, c_zips = set(housing_zip['postcode']), set(crashes_zip['zipcode'])
print('ZIPs in both:', len(h_zips & c_zips), '| only housing:', len(h_zips - c_zips),
      '| only crashes:', len(c_zips - h_zips))
print('Crashes in ZIPs with no affordable housing:',
      int(crashes_zip.loc[crashes_zip['zipcode'].isin(c_zips - h_zips), 'crashes'].sum()))

# Full OUTER join: a ZIP missing from one dataset genuinely has zero records there
# (no affordable building / no serious crash), and those ZIPs are the comparison group
# our question needs. An inner join would silently drop them.
merged_zip = housing_zip.merge(crashes_zip.rename(columns={'zipcode': 'postcode'}),
                               on='postcode', how='outer', indicator=True)
print(merged_zip['_merge'].value_counts())

ZIPs in both: 156 | only housing: 2 | only crashes: 31
Crashes in ZIPs with no affordable housing: 462
_merge
both          156
right_only     31
left_only       2
Name: count, dtype: int64


## 6. Post-integration cleaning
### 6.1–6.3 New missing values, type mismatches, redundant columns
- **Missing values from the join:** the 31 crash-only ZIPs have NaN in all housing columns and the 2 housing-only ZIPs have NaN in crash columns. These are structural zeros (no records), so we fill them with 0. Shares and rates that would divide by zero stay NaN on purpose.
- **Type mismatches:** counts came back as `float` after the outer join and are cast to `int`. ZIP is kept as a string in both tables (the raw files stored it as a float, `10001.0`).
- **Redundant columns:** the join indicator `_merge` is replaced by two readable flags (`in_housing`, `in_sirta`). The first version's duplicate `zipcode` column no longer exists because we rename before joining.

In [71]:
# ============================================================
# 6. POST-INTEGRATION CLEANING
# ============================================================
# 6.1 New missing values created by the join -> structural zeros (no records)
count_cols = ['buildings', 'total_units', 'counted_units', 'extremely_low_income_units',
              'very_low_income_units', 'low_income_units',
              'crashes', 'severe_injuries', 'fatalities', 'total_injured']
print('NaN created by the join:\n', merged_zip[count_cols].isna().sum())
merged_zip[count_cols] = merged_zip[count_cols].fillna(0)

# 6.2 Data-type mismatches: counts came back as float after the join
int_cols = ['buildings', 'crashes', 'fatalities', 'severe_injuries', 'total_injured']
merged_zip[int_cols] = merged_zip[int_cols].astype(int)
merged_zip['postcode'] = merged_zip['postcode'].astype(str)

# 6.3 Redundant columns: join indicator becomes a readable flag
merged_zip['in_housing'] = merged_zip['_merge'].isin(['both', 'left_only'])
merged_zip['in_sirta'] = merged_zip['_merge'].isin(['both', 'right_only'])
merged_zip = merged_zip.drop(columns='_merge')
print('\nDtypes after fix:\n', merged_zip.dtypes)

NaN created by the join:
 buildings                     31
total_units                   31
counted_units                 31
extremely_low_income_units    31
very_low_income_units         31
low_income_units              31
crashes                        2
severe_injuries                2
fatalities                     2
total_injured                  2
dtype: int64

Dtypes after fix:
 postcode                       object
buildings                       int64
total_units                   float64
counted_units                 float64
extremely_low_income_units    float64
very_low_income_units         float64
low_income_units              float64
new_construction_share        float64
crashes                         int64
severe_injuries                 int64
fatalities                      int64
total_injured                   int64
in_housing                       bool
in_sirta                         bool
dtype: object


### 6.4 Inconsistent borough labels
Some ZIPs straddle a borough line (e.g. 10463 includes Marble Hill; 11237 and 11385 sit on the Brooklyn/Queens border). The first version took the `first` borough it saw, which made 11237 "Brooklyn" in housing and "Queens" in crashes. We now assign the borough that holds **the most records from both datasets combined** in that ZIP. After this rule the two datasets agree on every shared ZIP.

In [72]:
# 6.4 Inconsistent borough labels - a few ZIPs cross a borough line, and the two datasets
#     can disagree. Rule: borough = the borough holding most records of BOTH datasets in that ZIP.
boro_votes = pd.concat([
    h[['postcode', 'borough']],
    s_zip[['zipcode', 'crash_borough']].rename(columns={'zipcode': 'postcode', 'crash_borough': 'borough'}),
]).dropna()
zip_borough = boro_votes.groupby('postcode')['borough'].agg(lambda b: b.value_counts().index[0])

h_boro = h.groupby('postcode')['borough'].agg(lambda b: b.value_counts().index[0])
c_boro = s_zip.dropna(subset=['crash_borough']).groupby('zipcode')['crash_borough'].agg(
    lambda b: b.value_counts().index[0])
both = pd.concat([h_boro.rename('housing'), c_boro.rename('sirta')], axis=1).dropna()
print('ZIPs where the two datasets disagree on borough:\n', both[both['housing'] != both['sirta']])

merged_zip['borough'] = merged_zip['postcode'].map(zip_borough)
print('ZIPs without a borough:', merged_zip['borough'].isna().sum())

ZIPs where the two datasets disagree on borough:
 Empty DataFrame
Columns: [housing, sirta]
Index: []
ZIPs without a borough: 0


### 6.5 ZIP centroids
Map positions are the mean coordinates of all housing buildings **and** crashes in the ZIP. The first version used housing buildings only, which pulled single-building ZIPs onto that one building and left no position for crash-only ZIPs.

In [73]:
# 6.5 ZIP centroid for the map: mean position of all housing buildings AND crashes in the ZIP
pts = pd.concat([
    h[['postcode', 'latitude', 'longitude']],
    s_zip[['zipcode', 'latitude', 'longitude']].rename(columns={'zipcode': 'postcode'}),
]).dropna()
centroids = pts.groupby('postcode')[['latitude', 'longitude']].mean().rename(
    columns={'latitude': 'lat', 'longitude': 'lon'})
merged_zip = merged_zip.join(centroids, on='postcode')
print('ZIPs without coordinates:', merged_zip['lat'].isna().sum())

ZIPs without coordinates: 0


### 6.6 Derived metrics and ratio outliers
`crashes_per_1000_units` explodes when a ZIP has almost no affordable housing: ZIP 11417 has **1** affordable unit and 73 crashes, giving 73,000 crashes per 1,000 units. That number is an artifact of a tiny denominator, not a finding. **Domain rule:** a rate is only computed for ZIPs with ≥ 100 affordable units (128 of 189 ZIPs). Among those, the IQR rule flags 23 high-rate ZIPs across four boroughs, typically commercial or arterial ZIPs with modest housing (e.g. Midtown East 10017/10022, Bensonhurst 11214, Elmhurst 11373). They are **flagged (`rate_outlier`) but kept**, because they are real places that the safety story should highlight.

In [74]:
# 6.6 Derived metrics (domain rule: rates only where the denominator is meaningful)
eligible = merged_zip['counted_units'] >= MIN_UNITS_FOR_RATE
merged_zip['crashes_per_1000_units'] = np.where(
    eligible, merged_zip['crashes'] / merged_zip['counted_units'] * 1000, np.nan)
merged_zip['eli_share'] = np.where(
    merged_zip['counted_units'] > 0,
    merged_zip['extremely_low_income_units'] / merged_zip['counted_units'] * 100, np.nan)
merged_zip['new_construction_share'] = merged_zip['new_construction_share'] * 100

# Why the threshold matters: the unrestricted ratio explodes for tiny denominators
raw_ratio = merged_zip['crashes'] / merged_zip['counted_units'].replace(0, np.nan) * 1000
print('Unrestricted ratio - max:', round(raw_ratio.max()), 'crashes per 1,000 units')
print(merged_zip.assign(raw_ratio=raw_ratio).nlargest(5, 'raw_ratio')[
    ['postcode', 'borough', 'counted_units', 'crashes', 'raw_ratio']])
print(f'\nZIPs with a rate (>= {MIN_UNITS_FOR_RATE} affordable units): {eligible.sum()} of {len(merged_zip)}')

# IQR outliers on the restricted rate - flagged, not removed
r = merged_zip['crashes_per_1000_units']
q1, q3 = r.quantile([0.25, 0.75])
merged_zip['rate_outlier'] = r > q3 + 1.5 * (q3 - q1)
print('Rate outliers (IQR):', merged_zip.loc[merged_zip['rate_outlier'], 'postcode'].tolist())

Unrestricted ratio - max: 73000 crashes per 1,000 units
    postcode borough  counted_units  crashes  raw_ratio
168    11417  Queens            1.0       73    73000.0
173    11422  Queens            1.0       33    33000.0
176    11427  Queens            1.0       16    16000.0
171    11420  Queens           10.0      103    10300.0
153    11370  Queens            5.0       44     8800.0

ZIPs with a rate (>= 100 affordable units): 128 of 189
Rate outliers (IQR): ['10013', '10017', '10021', '10022', '10028', '10465', '10466', '10471', '11214', '11218', '11219', '11223', '11229', '11230', '11232', '11234', '11354', '11369', '11372', '11373', '11385', '11412', '11418']


In [75]:
# 6.7 Final check and column order
merged_zip = merged_zip[[
    'postcode', 'borough', 'lat', 'lon', 'in_housing', 'in_sirta',
    'buildings', 'total_units', 'counted_units', 'extremely_low_income_units',
    'very_low_income_units', 'low_income_units', 'eli_share', 'new_construction_share',
    'crashes', 'severe_injuries', 'fatalities', 'total_injured',
    'crashes_per_1000_units', 'rate_outlier']].sort_values('postcode').reset_index(drop=True)
print(merged_zip.shape)
print('Remaining NaN (expected only in rate/share columns):\n', merged_zip.isna().sum()[lambda x: x > 0])
merged_zip.head()

(189, 20)
Remaining NaN (expected only in rate/share columns):
 eli_share                 31
new_construction_share    31
crashes_per_1000_units    61
dtype: int64


,postcode,borough,lat,lon,in_housing,in_sirta,buildings,total_units,counted_units,extremely_low_income_units,very_low_income_units,low_income_units,eli_share,new_construction_share,crashes,severe_injuries,fatalities,total_injured,crashes_per_1000_units,rate_outlier
0,10001,Manhattan,40.750320,-73.996151,True,True,22,5784.0,2670.0,429.0,434.0,1012.0,16.067416,38.801498,87,89,3,124,32.584270,False
1,10002,Manhattan,40.716522,-73.987609,True,True,64,8227.0,6744.0,982.0,2780.0,2124.0,14.561091,22.301305,124,130,10,158,18.386714,False
2,10003,Manhattan,40.731367,-73.989613,True,True,27,1009.0,714.0,363.0,167.0,163.0,50.840336,4.481793,76,75,4,80,106.442577,False
3,10004,Manhattan,40.704325,-74.012151,True,True,2,1891.0,473.0,0.0,95.0,80.0,0.000000,100.000000,10,10,0,11,21.141649,False
4,10005,Manhattan,40.705691,-74.007069,True,True,1,647.0,162.0,0.0,33.0,40.0,0.000000,100.000000,4,4,0,4,24.691358,False


### 6.8 Borough-level robustness table
Built from the **same ZIP sample** as `merged_zip`, so borough and ZIP results are directly comparable. The first version mixed the two: borough totals included crashes and buildings without a ZIP, and the chart text quoted numbers from the other table.

In [76]:
# ============================================================
# 6.8 Borough-level table (robustness check) - built from the SAME ZIP sample
# ============================================================
merged_boro = (merged_zip.groupby('borough')
               .agg(zips=('postcode', 'count'),
                    zips_with_housing=('in_housing', 'sum'),
                    buildings=('buildings', 'sum'),
                    counted_units=('counted_units', 'sum'),
                    total_units=('total_units', 'sum'),
                    crashes=('crashes', 'sum'),
                    fatalities=('fatalities', 'sum'),
                    total_injured=('total_injured', 'sum'))
               .reindex(BOROUGH_ORDER).reset_index())
merged_boro['crashes_per_1000_units'] = merged_boro['crashes'] / merged_boro['counted_units'] * 1000
merged_boro['units_share_%'] = merged_boro['counted_units'] / merged_boro['counted_units'].sum() * 100
merged_boro['crashes_share_%'] = merged_boro['crashes'] / merged_boro['crashes'].sum() * 100
merged_boro.round(1)

,borough,zips,zips_with_housing,buildings,counted_units,total_units,crashes,fatalities,total_injured,crashes_per_1000_units,units_share_%,crashes_share_%
0,Bronx,25,24,2169,103616.0,120388.0,2313,128,3035,22.3,33.7,20.3
1,Brooklyn,39,38,3027,86147.0,137065.0,3631,241,4627,42.1,28.0,31.8
2,Manhattan,53,42,1373,71614.0,104230.0,2395,131,2886,33.4,23.3,21.0
3,Queens,60,45,775,40852.0,61115.0,2706,160,3558,66.2,13.3,23.7
4,Staten Island,12,9,126,5236.0,6064.0,374,28,495,71.4,1.7,3.3


In [77]:
# Save the integrated tables (merged_zip/merged_boro for the deliverable,
# the two long tables power the website's construction-type and year filters)
merged_zip.to_csv('merged_zip.csv', index=False)
merged_boro.to_csv('merged_boro.csv', index=False)
housing_zip_type.to_csv('housing_zip_type.csv', index=False)
crashes_zip_year.rename(columns={'zipcode': 'postcode'}).to_csv('crashes_zip_year.csv', index=False)
print('Saved merged_zip.csv, merged_boro.csv, housing_zip_type.csv, crashes_zip_year.csv')

Saved merged_zip.csv, merged_boro.csv, housing_zip_type.csv, crashes_zip_year.csv


## 7. How strong is the relationship? (correlation analysis)
Both variables are heavily skewed and a few ZIPs are extreme, so we use **Spearman's rank correlation** (ρ), which measures whether ZIPs with more housing tend to rank higher in crashes without being dominated by outliers.

In [78]:
# ============================================================
# 7. How strong is the housing-crash relationship? (Spearman rank correlation)
# ============================================================
def spearman(df, x, y):
    rho, p = stats.spearmanr(df[x], df[y])
    return pd.Series({'n': len(df), 'rho': round(rho, 2), 'p_value': round(p, 4)})

hz = merged_zip[merged_zip['in_housing']]
rows = {
    'All ZIPs (incl. 0 units)': spearman(merged_zip, 'counted_units', 'crashes'),
    'ZIPs with housing': spearman(hz, 'counted_units', 'crashes'),
    'ZIPs with housing - fatalities': spearman(hz, 'counted_units', 'fatalities'),
}
for b in BOROUGH_ORDER:
    rows[f'  {b}'] = spearman(hz[hz['borough'] == b], 'counted_units', 'crashes')
corr_table = pd.DataFrame(rows).T
corr_table

,n,rho,p_value
All ZIPs (incl. 0 units),189.0,0.66,0.0000
ZIPs with housing,158.0,0.52,0.0000
ZIPs with housing - fatalities,158.0,0.35,0.0000
Bronx,24.0,0.56,0.0044
Brooklyn,38.0,0.29,0.0752
Manhattan,42.0,0.55,0.0002
Queens,45.0,0.25,0.1010
Staten Island,9.0,0.53,0.1392


**Result — the relationship is moderate and positive, not absent.** Across the 158 ZIPs with affordable housing, ρ = **0.52** (p < 0.001). Including the 31 no-housing ZIPs it is 0.66, and for fatalities 0.35. Within boroughs it is ρ ≈ 0.55 in the Bronx and Manhattan but weaker and not significant in Brooklyn (0.29, p = 0.08) and Queens (0.25, p = 0.10). Staten Island (n = 9) is too small to test.

**Interpretation:** the first version's headline ("no relationship") was wrong. A likely explanation is a common cause: ZIPs with more affordable housing are dense, populous neighbourhoods with more people, more traffic and more arterial streets, so they have more of *both*. Without population or traffic-volume data we **cannot** separate these explanations, and this is correlation, not causation.

## 8. Visualizations
### Chart 1 — RQ1: Do ZIPs with more affordable units have more serious crashes?

In [79]:
# ---- Chart 1 (RQ1) ----
rho1 = corr_table.loc['ZIPs with housing', 'rho']
fig1 = px.scatter(
    hz, x='counted_units', y='crashes', color='borough', size='fatalities', size_max=18,
    hover_name='postcode', hover_data={'counted_units': ':,', 'crashes': True, 'fatalities': True},
    log_x=True, category_orders={'borough': BOROUGH_ORDER}, color_discrete_map=BOROUGH_COLORS,
    title=f'Chart 1 - Affordable units vs. serious crashes per ZIP (Spearman rho = {rho1})',
    labels={'counted_units': 'Affordable units in ZIP (log scale)', 'crashes': 'Serious-injury crashes 2022-2026',
            'borough': 'Borough', 'fatalities': 'Fatalities'})
fig1.update_layout(template=TEMPLATE, height=600)
fig1.show()
fig1.write_html('chart01_units_vs_crashes.html', include_plotlyjs='cdn')

**What we see:** the cloud slopes upward (ρ = 0.52), but with wide spread. 11236 (Brooklyn: 48 affordable units, 180 crashes) and 10475 (Bronx / Co-op City: 15,534 units, 44 crashes) are the clearest exceptions. Bronx and Manhattan points follow the trend most closely; Brooklyn and Queens points are scattered.

**Answer to RQ1:** yes, moderately: ZIPs with more affordable housing tend to have more serious crashes, but housing volume alone explains only part of the ranking.

### Chart 2 — RQ2: Which borough carries the heaviest crash burden relative to its affordable housing?

In [80]:
# ---- Chart 2 (RQ2) ----
fig2 = px.bar(merged_boro, x='borough', y='crashes_per_1000_units', color='borough',
              color_discrete_map=BOROUGH_COLORS, text_auto='.1f',
              hover_data={'crashes': ':,', 'counted_units': ':,', 'zips': True},
              title='Chart 2 - Serious crashes per 1,000 affordable units, by borough',
              labels={'crashes_per_1000_units': 'Crashes per 1,000 affordable units',
                      'borough': 'Borough', 'counted_units': 'Affordable units', 'zips': 'ZIPs'})
fig2.add_hline(y=merged_boro['crashes'].sum() / merged_boro['counted_units'].sum() * 1000,
               line_dash='dash', annotation_text='Citywide')
fig2.update_layout(template=TEMPLATE, height=500, showlegend=False)
fig2.show()
fig2.write_html('chart02_borough_rate.html', include_plotlyjs='cdn')
merged_boro[['borough', 'counted_units', 'crashes', 'crashes_per_1000_units']].round(1)

,borough,counted_units,crashes,crashes_per_1000_units
0,Bronx,103616.0,2313,22.3
1,Brooklyn,86147.0,3631,42.1
2,Manhattan,71614.0,2395,33.4
3,Queens,40852.0,2706,66.2
4,Staten Island,5236.0,374,71.4


**What we see:** Staten Island (71.4) and Queens (66.2) have about three times the Bronx's rate (22.3) of serious crashes per 1,000 affordable units. Brooklyn (42.1) is slightly above the citywide 37.1 and Manhattan (33.4) slightly below.

**Answer to RQ2:** Staten Island has the highest burden per unit, with Queens close behind. (The first version said Queens; that came from mixing two tables.) Staten Island rests on only 12 ZIPs and 5,236 units, so its rate is the least certain. The pattern is consistent with car-oriented outer boroughs having more serious crashes per resident, but we have no population data to confirm it.

**Limitation:** the rate uses affordable units as the denominator, not total population, so it measures *crash exposure relative to affordable-housing investment*, not risk per person.

### Chart 3 — RQ3: Are the most dangerous ZIPs also major affordable-housing locations?

In [81]:
# ---- Chart 3 (RQ3) ----
merged_zip['units_percentile'] = merged_zip['counted_units'].rank(pct=True) * 100
top_crash = merged_zip.nlargest(10, 'crashes').sort_values('crashes')
fig3 = px.bar(top_crash, x='crashes', y='postcode', orientation='h',
              color='units_percentile', color_continuous_scale='Viridis', range_color=[0, 100],
              text=top_crash['units_percentile'].round(0).astype(int).astype(str) + 'th pct',
              hover_data={'borough': True, 'counted_units': ':,', 'fatalities': True},
              title='Chart 3 - Top 10 crash ZIPs, coloured by their affordable-housing percentile',
              labels={'crashes': 'Serious-injury crashes', 'postcode': 'ZIP code',
                      'units_percentile': 'Housing percentile', 'counted_units': 'Affordable units'})
fig3.update_layout(template=TEMPLATE, height=500, yaxis_type='category')
fig3.show()
fig3.write_html('chart03_top_crash_zips.html', include_plotlyjs='cdn')
top_crash[['postcode', 'borough', 'crashes', 'counted_units', 'units_percentile']].iloc[::-1].round(0)

,postcode,borough,crashes,counted_units,units_percentile
104,11207,Brooklyn,180,5080.0,91.0
132,11236,Brooklyn,180,48.0,26.0
109,11212,Brooklyn,161,6351.0,94.0
70,10456,Bronx,156,6825.0,96.0
100,11203,Brooklyn,154,2069.0,73.0
67,10453,Bronx,148,7110.0,97.0
103,11206,Brooklyn,145,6225.0,93.0
81,10467,Bronx,145,4520.0,89.0
71,10457,Bronx,144,7625.0,97.0
130,11234,Brooklyn,140,159.0,38.0


**What we see:** 6 of the top-10 crash ZIPs are in Brooklyn and 4 in the Bronx. **8 of the 10 rank at or above the 73rd percentile for affordable housing**, and 7 rank at or above the 89th (e.g. 11207 East New York, 11212 Brownsville, 10456/10453/10457 in the south and west Bronx). The exceptions are 11236 (Canarsie, 26th percentile) and 11234 (Marine Park / Flatlands, 38th percentile). Both are car-oriented southeast-Brooklyn ZIPs with large arterials.

**Answer to RQ3:** mostly yes. The ZIPs with the most serious crashes are, with two exceptions, among the city's major affordable-housing neighbourhoods. (The first version's claim that they have "small housing footprints" was only true for 11236.)

### Chart 4 — RQ4: Do the ZIPs with the most affordable housing see above-median crashes?

In [82]:
# ---- Chart 4 (RQ4) ----
median_crashes = merged_zip['crashes'].median()
top_units = merged_zip.nlargest(10, 'counted_units').sort_values('counted_units')
top_units['vs_median'] = np.where(top_units['crashes'] > median_crashes, 'Above median', 'At/below median')
fig4 = px.bar(top_units, x='crashes', y='postcode', orientation='h', color='vs_median',
              color_discrete_map={'Above median': '#d62728', 'At/below median': '#7f7f7f'},
              text='crashes', hover_data={'borough': True, 'counted_units': ':,'},
              title='Chart 4 - Crashes in the 10 ZIPs with the most affordable units',
              labels={'crashes': 'Serious-injury crashes', 'postcode': 'ZIP (ordered by affordable units)',
                      'vs_median': '', 'counted_units': 'Affordable units'})
fig4.add_vline(x=median_crashes, line_dash='dash', annotation_text=f'Median ZIP: {median_crashes:.0f}')
fig4.update_layout(template=TEMPLATE, height=500, yaxis_type='category')
fig4.show()
fig4.write_html('chart04_top_housing_zips.html', include_plotlyjs='cdn')
top_units[['postcode', 'borough', 'counted_units', 'crashes', 'vs_median']].iloc[::-1]

,postcode,borough,counted_units,crashes,vs_median
89,10475,Bronx,15534.0,44,At/below median
135,11239,Brooklyn,9670.0,27,At/below median
66,10452,Bronx,9324.0,121,Above median
26,10029,Manhattan,8409.0,119,Above median
65,10451,Bronx,7974.0,127,Above median
71,10457,Bronx,7625.0,144,Above median
67,10453,Bronx,7110.0,148,Above median
185,11691,Queens,6858.0,48,At/below median
70,10456,Bronx,6825.0,156,Above median
74,10460,Bronx,6784.0,97,Above median


**What we see:** 7 of the 10 highest-housing ZIPs are above the citywide median of 54 crashes, mostly in the Bronx. The three exceptions are revealing: **10475** (Co-op City), **11239** (Starrett City / Spring Creek) and **11691** (Far Rockaway). All three are large self-contained housing complexes at the edge of the city, with internal streets and less through-traffic.

**Answer to RQ4:** usually yes, but large planned complexes at the city edge are the exception. That suggests the *street setting* of housing matters more than the number of units. (This is a hypothesis; we have no street-network data to test it.)

### Chart 5 — RQ5: Has the crash burden per affordable unit changed from 2022 to 2026?

In [83]:
# ---- Chart 5 (RQ5) ----
cy = crashes_zip_year.merge(merged_zip[['postcode', 'borough']], left_on='zipcode', right_on='postcode')
boro_year = cy.groupby(['borough', 'year'])['crashes'].sum().reset_index()
boro_year['annualised'] = boro_year['crashes'] / boro_year['year'].map(YEAR_FRACTION)
boro_year = boro_year.merge(merged_boro[['borough', 'counted_units']], on='borough')
boro_year['rate'] = boro_year['annualised'] / boro_year['counted_units'] * 1000
heat = boro_year.pivot(index='borough', columns='year', values='rate').reindex(BOROUGH_ORDER)
heat.columns = [f'{y}*' if y == 2026 else str(y) for y in heat.columns]
fig5 = px.imshow(heat, text_auto='.1f', color_continuous_scale='Reds', aspect='auto',
                 title='Chart 5 - Crashes per 1,000 affordable units per year (*2026 annualised from Jan-Jun)',
                 labels={'x': 'Crash year', 'y': 'Borough', 'color': 'Crashes per 1,000 units'})
fig5.update_layout(template=TEMPLATE, height=420)
fig5.show()
fig5.write_html('chart05_rate_heatmap.html', include_plotlyjs='cdn')
heat.round(1)

,2022,2023,2024,2025,2026*
borough,,,,,
Bronx,5.0,5.0,5.1,4.9,4.5
Brooklyn,8.9,9.4,9.6,9.8,8.8
Manhattan,6.8,7.9,7.6,7.4,7.4
Queens,12.4,14.4,15.4,17.2,13.6
Staten Island,13.6,18.3,14.7,13.6,22.7


**What we see:** the Bronx is stable and lowest (≈ 5 crashes per 1,000 units per year). Brooklyn and Manhattan are stable at ≈ 7–10. **Queens rises every full year from 12.4 (2022) to 17.2 (2025)**, a 39 % increase, before an annualised 13.6 in 2026. Staten Island swings between 13.6 and 22.7 because it has few crashes.

**Answer to RQ5:** the gap between boroughs is persistent, and Queens is the only borough with a clear upward trend through 2025. The 2026 values are extrapolated from six months (winter/spring months may have fewer crashes than summer), so they should not be over-read. Housing units are held fixed, so a changing rate reflects changing crash counts.

### Chart 6 — RQ7: Do crash hotspots and affordable-housing concentrations overlap in space?

In [84]:
# ---- Chart 6 (RQ7) ----
md = merged_zip.dropna(subset=['lat', 'lon']).copy()
md['units_for_colour'] = md['counted_units'].clip(upper=md['counted_units'].quantile(0.95))
fig6 = px.scatter_map(
    md, lat='lat', lon='lon', size='crashes', size_max=22, color='units_for_colour',
    color_continuous_scale='Viridis', hover_name='postcode',
    hover_data={'borough': True, 'counted_units': ':,', 'crashes': True, 'fatalities': True,
                'lat': False, 'lon': False, 'units_for_colour': False},
    zoom=9.3, center={'lat': 40.71, 'lon': -73.93}, map_style='carto-positron', height=650,
    title='Chart 6 - Bubble size = serious crashes, colour = affordable units (capped at 95th pct)',
    labels={'units_for_colour': 'Affordable units', 'counted_units': 'Affordable units'})
fig6.update_layout(margin=dict(l=0, r=0, t=50, b=0))
fig6.show()
fig6.write_html('chart06_map.html', include_plotlyjs='cdn')

**What we see:** the biggest bubbles (most crashes) sit in central/eastern Brooklyn (East New York, Brownsville, Canarsie, Flatbush) and the south and west Bronx. Several of these are also the brightest (most affordable housing), especially the south Bronx and East New York. Bright but small bubbles mark the large edge complexes from Chart 4 (Co-op City, Starrett City, the Rockaways). Eastern Queens and Staten Island show many mid-size bubbles with dark (low-housing) colour.

In [85]:
# Quantify the overlap shown on the map: how many of the top-20 crash ZIPs are also top-20 housing ZIPs?
top20_crash = set(merged_zip.nlargest(20, 'crashes')['postcode'])
top20_units = set(merged_zip.nlargest(20, 'counted_units')['postcode'])
print('Overlap of top-20 crash ZIPs and top-20 housing ZIPs:', len(top20_crash & top20_units), sorted(top20_crash & top20_units))
print('Top-20 crash ZIPs by borough:', merged_zip[merged_zip['postcode'].isin(top20_crash)]['borough'].value_counts().to_dict())

Overlap of top-20 crash ZIPs and top-20 housing ZIPs: 10 ['10002', '10451', '10452', '10453', '10456', '10457', '11206', '11207', '11208', '11212']
Top-20 crash ZIPs by borough: {'Brooklyn': 9, 'Bronx': 8, 'Manhattan': 2, 'Queens': 1}


**Answer to RQ7:** partly. **10 of the 20 highest-crash ZIPs are also among the 20 highest-housing ZIPs**, all in the Bronx, central Brooklyn and the Lower East Side (10002). The other half of the crash hotspots are low-housing ZIPs, mostly in southeast Brooklyn and Queens.

**Limitation:** bubbles sit at ZIP centroids, not at crash locations, and ZIP size varies a lot.

### Chart 7 — RQ8: Do ZIPs with different amounts of affordable housing follow different crash trends?

In [86]:
# ---- Chart 7 (RQ8) ----
# Group ZIPs by how much affordable housing they have, then follow crashes per ZIP over time
hz_terc = merged_zip.copy()
hz_terc['housing_group'] = 'No affordable housing'
has = hz_terc['counted_units'] > 0
hz_terc.loc[has, 'housing_group'] = pd.qcut(hz_terc.loc[has, 'counted_units'], 3,
                                            labels=['Low housing', 'Medium housing', 'High housing']).astype(str)
grp = (crashes_zip_year.merge(hz_terc[['postcode', 'housing_group']], left_on='zipcode', right_on='postcode')
       .groupby(['housing_group', 'year'])['crashes'].sum().reset_index())
grp['zips'] = grp['housing_group'].map(hz_terc['housing_group'].value_counts())
grp['crashes_per_zip'] = grp['crashes'] / grp['year'].map(YEAR_FRACTION) / grp['zips']
group_order = ['No affordable housing', 'Low housing', 'Medium housing', 'High housing']
fig7 = px.line(grp, x='year', y='crashes_per_zip', color='housing_group', markers=True,
               category_orders={'housing_group': group_order},
               title='Chart 7 - Serious crashes per ZIP per year, by affordable-housing group (2026 annualised)',
               labels={'year': 'Year', 'crashes_per_zip': 'Crashes per ZIP per year',
                       'housing_group': 'ZIP group (terciles of affordable units)'})
fig7.update_layout(template=TEMPLATE, height=500, xaxis=dict(dtick=1))
fig7.show()
fig7.write_html('chart07_trend_by_housing_group.html', include_plotlyjs='cdn')
print(hz_terc['housing_group'].value_counts())
grp.pivot(index='housing_group', columns='year', values='crashes_per_zip').reindex(group_order).round(1)

housing_group
High housing             53
Low housing              53
Medium housing           52
No affordable housing    31
Name: count, dtype: int64


year,2022,2023,2024,2025,2026
housing_group,,,,,
No affordable housing,2.2,3.1,3.6,3.8,4.3
Low housing,9.5,10.8,11.2,11.5,10.6
Medium housing,12.7,14.6,14.8,15.2,13.7
High housing,21.2,21.8,21.4,21.5,19.3


**What we see:** ZIPs are split into four groups (no housing, then terciles of affordable units). The ordering never changes: high-housing ZIPs average ≈ 21 serious crashes per year, medium ≈ 13–15, low ≈ 10–11, no-housing ≈ 2–4. All groups are roughly flat over 2022–2025. The no-housing group rises most in relative terms (2.2 → 3.8, about +70 %), but from a very small base.

**Answer to RQ8:** the gap is structural, not a trend: high-housing ZIPs had about twice the crashes of low-housing ZIPs in every year, and the gap did not widen or close.

### Chart 8 — RQ6: Is each borough's share of crashes proportional to its share of affordable units?

In [87]:
# ---- Chart 8 (RQ6) ----
share_long = merged_boro.melt(id_vars='borough', value_vars=['units_share_%', 'crashes_share_%'],
                              var_name='measure', value_name='share')
share_long['measure'] = share_long['measure'].map({'units_share_%': 'Share of affordable units',
                                                   'crashes_share_%': 'Share of serious crashes'})
fig8 = px.bar(share_long, x='borough', y='share', color='measure', barmode='group', text_auto='.1f',
              title='Chart 8 - Each borough\'s share of affordable units vs. share of serious crashes',
              labels={'share': 'Share of city total (%)', 'borough': 'Borough', 'measure': ''})
fig8.update_layout(template=TEMPLATE, height=500)
fig8.show()
fig8.write_html('chart08_share_compare.html', include_plotlyjs='cdn')
merged_boro.assign(ratio=merged_boro['crashes_share_%'] / merged_boro['units_share_%'])[
    ['borough', 'units_share_%', 'crashes_share_%', 'ratio']].round(2)

,borough,units_share_%,crashes_share_%,ratio
0,Bronx,33.70,20.26,0.60
1,Brooklyn,28.02,31.80,1.13
2,Manhattan,23.29,20.97,0.90
3,Queens,13.29,23.70,1.78
4,Staten Island,1.70,3.28,1.92


**What we see:** if crashes tracked affordable housing, the bars in each pair would be equal (ratio = 1). The Bronx holds **33.7 %** of affordable units but only **20.3 %** of serious crashes (ratio 0.60). Queens holds **13.3 %** of units but **23.7 %** of crashes (1.78). Staten Island is 1.7 % vs 3.3 % (1.92). Brooklyn (28.0 % vs 31.8 %, 1.13) and Manhattan (23.3 % vs 21.0 %, 0.90) are close to proportional.

**Answer to RQ6:** no. Crash burden is **disproportionately high in Queens and Staten Island** and **disproportionately low in the Bronx** relative to where affordable housing is. For Vision Zero this means a housing-weighted allocation of safety money would under-serve Queens.

### Chart 9 — RQ9: Do ZIPs whose affordable housing targets the lowest incomes have higher crash rates?

In [88]:
# ---- Chart 9 (RQ9) ----
rated = merged_zip[merged_zip['counted_units'] >= MIN_UNITS_FOR_RATE]
rho9, p9 = stats.spearmanr(rated['eli_share'], rated['crashes_per_1000_units'])
fig9 = px.scatter(rated, x='eli_share', y='crashes_per_1000_units', color='borough',
                  size='counted_units', size_max=25, log_y=True, hover_name='postcode',
                  category_orders={'borough': BOROUGH_ORDER}, color_discrete_map=BOROUGH_COLORS,
                  title=f'Chart 9 - Extremely-low-income share vs. crash rate (rho = {rho9:.2f}, p = {p9:.3f})',
                  labels={'eli_share': 'Extremely-low-income units (% of affordable units)',
                          'crashes_per_1000_units': 'Crashes per 1,000 affordable units (log)',
                          'borough': 'Borough', 'counted_units': 'Affordable units'})
fig9.update_layout(template=TEMPLATE, height=550)
fig9.show()
fig9.write_html('chart09_income_vs_rate.html', include_plotlyjs='cdn')
print(f'n = {len(rated)} ZIPs, Spearman rho = {rho9:.3f}, p = {p9:.4f}')
print('Same test within boroughs:')
for b in BOROUGH_ORDER:
    sub = rated[rated['borough'] == b]
    if len(sub) >= 5:
        print(f'  {b:<14} n={len(sub):>2} rho={stats.spearmanr(sub["eli_share"], sub["crashes_per_1000_units"])[0]:.2f}')

n = 128 ZIPs, Spearman rho = -0.052, p = 0.5633
Same test within boroughs:
  Bronx          n=24 rho=-0.22
  Brooklyn       n=34 rho=-0.03
  Manhattan      n=37 rho=0.09
  Queens         n=28 rho=0.08
  Staten Island  n= 5 rho=0.00


**What we see:** no trend. Across the 128 ZIPs with ≥ 100 affordable units, the share of extremely-low-income (ELI) units is unrelated to the crash rate: ρ = −0.05, p = 0.56. The within-borough correlations are also near zero (−0.22 to 0.09).

**Answer to RQ9:** no. Within the affordable-housing stock, ZIPs that serve the poorest households do not have higher (or lower) crash rates per unit. (The first version tested this on only 5 borough totals, which cannot support any conclusion.)

### Chart 10 — RQ10: New construction vs preservation — different crash rates?

In [89]:
# ---- Chart 10 (RQ10) ----
rated = rated.copy()
rated['dominant_type'] = np.where(rated['new_construction_share'] >= 50,
                                  'Mostly new construction', 'Mostly preservation')
fig10 = px.box(rated, x='dominant_type', y='crashes_per_1000_units', color='dominant_type',
               points='all', log_y=True, hover_name='postcode',
               hover_data={'borough': True, 'counted_units': ':,', 'crashes': True},
               title='Chart 10 - Crash rate in ZIPs dominated by new construction vs. preservation',
               labels={'dominant_type': 'Majority of affordable units are...',
                       'crashes_per_1000_units': 'Crashes per 1,000 affordable units (log)',
                       'counted_units': 'Affordable units'})
fig10.update_layout(template=TEMPLATE, height=520, showlegend=False)
fig10.show()
fig10.write_html('chart10_construction_type.html', include_plotlyjs='cdn')

a = rated.loc[rated['dominant_type'] == 'Mostly new construction', 'crashes_per_1000_units']
b = rated.loc[rated['dominant_type'] == 'Mostly preservation', 'crashes_per_1000_units']
u, p10 = stats.mannwhitneyu(a, b)
print(rated.groupby('dominant_type')['crashes_per_1000_units'].describe()[['count', '50%', 'mean']].round(1))
print(f'Mann-Whitney U = {u:.0f}, p = {p10:.4f}')

# Is the difference real, or caused by which boroughs fall into each group?
print('\nZIPs per group and borough:')
print(pd.crosstab(rated['borough'], rated['dominant_type']))
print('\nSame test within each borough:')
for bname, g in rated.groupby('borough'):
    ga = g.loc[g['dominant_type'] == 'Mostly new construction', 'crashes_per_1000_units']
    gb = g.loc[g['dominant_type'] == 'Mostly preservation', 'crashes_per_1000_units']
    if len(ga) >= 3 and len(gb) >= 3:
        print(f'  {bname:<10} median new {ga.median():5.1f} vs preservation {gb.median():5.1f}  '
              f'p = {stats.mannwhitneyu(ga, gb)[1]:.2f}')

                         count   50%   mean
dominant_type                              
Mostly new construction   58.0  45.6  125.2
Mostly preservation       70.0  35.0   84.5
Mann-Whitney U = 2458, p = 0.0407

ZIPs per group and borough:
dominant_type  Mostly new construction  Mostly preservation
borough                                                    
Bronx                               11                   13
Brooklyn                            21                   13
Manhattan                           10                   27
Queens                              15                   13
Staten Island                        1                    4

Same test within each borough:
  Bronx      median new  32.0 vs preservation  28.4  p = 0.73
  Brooklyn   median new  47.5 vs preservation  55.3  p = 0.86
  Manhattan  median new  91.2 vs preservation  36.3  p = 0.08
  Queens     median new  68.8 vs preservation  40.0  p = 0.38


In [90]:
# How well are FATAL crashes covered compared with non-fatal ones?
fatal = s['total_fatalities'] > 0
print(f'Fatal crashes: {fatal.sum():,}')
print(f"Have coordinates:   fatal {s.loc[fatal, 'latitude'].notna().mean():.1%}  vs  non-fatal {s.loc[~fatal, 'latitude'].notna().mean():.1%}")
print(f"In the ZIP analysis: fatal {s.loc[fatal, 'zipcode'].notna().mean():.1%}  vs  non-fatal {s.loc[~fatal, 'zipcode'].notna().mean():.1%}")

Fatal crashes: 1,047
Have coordinates:   fatal 19.7%  vs  non-fatal 86.9%
In the ZIP analysis: fatal 63.0%  vs  non-fatal 87.7%


In [91]:
# ---- Chart 11 (RQ11) ----
# When do serious crashes happen? Compare the hour-of-day profile of crashes in
# high-housing ZIPs with all other ZIPs (housing groups from Chart 7).
s_time = s_zip.merge(hz_terc[['postcode', 'housing_group']], left_on='zipcode', right_on='postcode')
s_time['hour'] = pd.to_datetime(s_time['crash_time'], format='%H:%M', errors='coerce').dt.hour
print('Crashes without a parseable time:', s_time['hour'].isna().sum())
s_time = s_time.dropna(subset=['hour'])
s_time['hour'] = s_time['hour'].astype(int)
s_time['group2'] = np.where(s_time['housing_group'] == 'High housing',
                            'High-housing ZIPs', 'All other ZIPs')

hourly = (s_time.groupby(['group2', 'hour']).size()
          .groupby(level=0).transform(lambda x: x / x.sum() * 100)
          .reset_index(name='pct'))
fig11 = px.line(hourly, x='hour', y='pct', color='group2', markers=True,
                title='Chart 11 - Hour of day of serious crashes: high-housing ZIPs vs. all other ZIPs',
                labels={'hour': 'Hour of day (0 = midnight)', 'pct': 'Share of the group\'s crashes (%)',
                        'group2': ''})
fig11.update_layout(template=TEMPLATE, height=480, xaxis=dict(dtick=2))
fig11.show()
fig11.write_html('chart11_hour_of_day.html', include_plotlyjs='cdn')

# Summary periods + chi-square test: is the timing distribution different between the groups?
periods = pd.cut(s_time['hour'], bins=[-1, 5, 9, 15, 19, 23],
                 labels=['Night 0-5', 'AM peak 6-9', 'Midday 10-15', 'PM peak 16-19', 'Evening 20-23'])
tab = pd.crosstab(s_time['group2'], periods)
chi2, p11, dof, _ = stats.chi2_contingency(tab)
print((tab.div(tab.sum(axis=1), axis=0) * 100).round(1))
print(f'Chi-square = {chi2:.1f}, dof = {dof}, p = {p11:.4f}')

Crashes without a parseable time: 0


hour               Night 0-5  AM peak 6-9  Midday 10-15  PM peak 16-19  Evening 20-23
group2                                                                               
All other ZIPs          15.1         13.3          28.7           25.0           17.9
High-housing ZIPs       16.6         12.6          26.2           24.0           20.5
Chi-square = 22.9, dof = 4, p = 0.0001


In [92]:
# ---- Chart 12 (RQ12) ----
# Housing timing x crash trend: did ZIPs that received the most NEWLY COMPLETED affordable
# units in 2022-2025 see serious crashes rise faster than other ZIPs?
completed = h[h['building_completion_date'].dt.year.between(2022, 2025)]
print(f"Buildings completed 2022-2025: {len(completed):,} ({int(completed['all_counted_units'].sum()):,} affordable units); "
      f"buildings with no completion date (still in progress): {h['building_completion_date'].isna().sum():,}")
added = completed.groupby('postcode')['all_counted_units'].sum()

cz = crashes_zip_year.pivot_table(index='zipcode', columns='year', values='crashes', aggfunc='sum').fillna(0)
trend = merged_zip[['postcode', 'borough']].copy()
trend['units_added'] = trend['postcode'].map(added).fillna(0)
trend['crashes_2022'] = trend['postcode'].map(cz[2022]).fillna(0)
trend['crashes_2025'] = trend['postcode'].map(cz[2025]).fillna(0)   # 2025 = last full year
# Domain rule: ignore ZIPs with < 10 crashes across both years (changes would be noise)
trend = trend[(trend['crashes_2022'] + trend['crashes_2025']) >= 10].copy()
trend['change'] = trend['crashes_2025'] - trend['crashes_2022']

med_added = trend.loc[trend['units_added'] > 0, 'units_added'].median()
trend['added_group'] = np.select(
    [trend['units_added'] == 0, trend['units_added'] < med_added],
    ['No new units', f'Some new units (< {med_added:,.0f})'], default=f'Many new units (>= {med_added:,.0f})')
grp12 = (trend.groupby('added_group')
         .agg(zips=('postcode', 'count'), crashes_2022=('crashes_2022', 'sum'),
              crashes_2025=('crashes_2025', 'sum'))
         .reset_index())
grp12['pct_change'] = (grp12['crashes_2025'] / grp12['crashes_2022'] - 1) * 100
order12 = ['No new units', f'Some new units (< {med_added:,.0f})', f'Many new units (>= {med_added:,.0f})']

fig12 = px.bar(grp12, x='added_group', y='pct_change', color='added_group', text_auto='.1f',
               category_orders={'added_group': order12},
               hover_data={'zips': True, 'crashes_2022': ':,', 'crashes_2025': ':,'},
               title='Chart 12 - Change in serious crashes 2022 -> 2025, by affordable units completed 2022-2025',
               labels={'added_group': 'Affordable units completed in the ZIP, 2022-2025',
                       'pct_change': 'Change in serious crashes, 2022 to 2025 (%)',
                       'zips': 'ZIPs', 'crashes_2022': 'Crashes 2022', 'crashes_2025': 'Crashes 2025'})
fig12.add_hline(y=0, line_color='black')
fig12.update_layout(template=TEMPLATE, height=500, showlegend=False)
fig12.show()
fig12.write_html('chart12_new_housing_vs_crash_change.html', include_plotlyjs='cdn')

print(grp12.set_index('added_group').reindex(order12).round(1))
rho12, p12 = stats.spearmanr(trend['units_added'], trend['change'])
print(f'\nZIP level (n = {len(trend)}): Spearman rho(units added, change in crashes) = {rho12:.2f}, p = {p12:.3f}')
print('Within boroughs:')
for bname, g in trend.groupby('borough'):
    if len(g) >= 5:
        print(f'  {bname:<14} n={len(g):>2}  rho = {stats.spearmanr(g["units_added"], g["change"])[0]:.2f}')

Buildings completed 2022-2025: 2,234 (98,629 affordable units); buildings with no completion date (still in progress): 1,586


                         zips  crashes_2022  crashes_2025  pct_change
added_group                                                          
No new units               20         170.0         232.0        36.5
Some new units (< 320)     62         847.0        1005.0        18.7
Many new units (>= 320)    63        1285.0        1331.0         3.6

ZIP level (n = 145): Spearman rho(units added, change in crashes) = -0.12, p = 0.168
Within boroughs:
  Bronx          n=23  rho = -0.29
  Brooklyn       n=38  rho = -0.06
  Manhattan      n=35  rho = -0.02
  Queens         n=42  rho = -0.04
  Staten Island  n= 7  rho = -0.17


In [93]:
# ============================================================
# Export the extra tables the website needs for Charts 11 and 12
# ============================================================
# Chart 11: crashes per ZIP x year x hour of day
s_hour = s_zip.copy()
s_hour['hour'] = pd.to_datetime(s_hour['crash_time'], format='%H:%M', errors='coerce').dt.hour
crashes_zip_hour = (s_hour.dropna(subset=['hour'])
                    .astype({'hour': int})
                    .groupby(['zipcode', 'year', 'hour'])
                    .agg(crashes=('crash_key', 'count'),
                         fatalities=('total_fatalities', 'sum'),
                         total_injured=('total_injured', 'sum'))
                    .reset_index()
                    .rename(columns={'zipcode': 'postcode'}))
crashes_zip_hour.to_csv('crashes_zip_hour.csv', index=False)

# Chart 12: affordable units in buildings COMPLETED 2022-2025, per ZIP x construction type
housing_completed = (h[h['building_completion_date'].dt.year.between(2022, 2025)]
                     .groupby(['postcode', 'construction_type'])['all_counted_units'].sum()
                     .reset_index(name='units_completed'))
housing_completed.to_csv('housing_completed_zip_type.csv', index=False)

print('crashes_zip_hour.csv:', crashes_zip_hour.shape, '| total crashes:', crashes_zip_hour['crashes'].sum())
print('housing_completed_zip_type.csv:', housing_completed.shape,
      '| units:', int(housing_completed['units_completed'].sum()))

crashes_zip_hour.csv: (7701, 6) | total crashes: 11419
housing_completed_zip_type.csv: (209, 3) | units: 98629


**What we see:** citywide, ZIPs where most affordable units are new construction have a higher median crash rate (45.6 vs 35.0 per 1,000 units; Mann-Whitney p = 0.04). **But the difference disappears within every borough** (all p ≥ 0.08), and in Brooklyn it reverses. The citywide gap is a **composition effect**: Manhattan's many low-rate ZIPs are mostly preservation (27 of 37), while high-rate Queens and Brooklyn ZIPs lean towards new construction.

**Answer to RQ10:** no evidence that new construction itself goes with more crashes. The apparent citywide difference is explained by borough. This is a good example of why we test within groups before drawing conclusions.

## 9. Story summary

**Question.** Do the neighbourhoods receiving the most affordable housing also experience the most serious traffic crashes?

**Data.** HPD's Affordable Housing Production by Building (7,470 deduplicated buildings, 307k affordable units) and SIRTA serious-injury crashes 2022 – Jun 2026 (11,419 crashes placed in a ZIP after nearest-neighbour recovery), integrated at the level of 189 ZIP codes with a full outer join.

**Key findings**
1. **They do co-locate.** ZIPs with more affordable housing have more serious crashes (Spearman ρ = 0.52). Half of the 20 highest-crash ZIPs are also top-20 housing ZIPs (the Bronx, East New York, Brownsville).
2. **But the burden per unit is very uneven.** Queens and Staten Island have about 3× the Bronx's crashes per 1,000 affordable units, and Queens' rate rose 39 % from 2022 to 2025.
3. **Large edge-of-city complexes are the exception.** Co-op City, Starrett City and Far Rockaway have very high housing but below-median crashes.
4. **What gets built doesn't matter much.** Neither the income mix (ρ ≈ 0) nor new construction vs preservation (no difference within boroughs) predicts the crash rate.

**What surprised us.** The original analysis concluded "no relationship". Fixing the data (comma-formatted unit counts, market-rate units counted as affordable, duplicated buildings, 40 % of crashes without ZIP) reversed that conclusion.

**Limitations**
- **Correlation, not causation.** Population, traffic volume and road design are not in our data and probably drive both housing density and crashes.
- **Denominator:** rates are per affordable unit, not per resident or per vehicle-mile.
- **Coverage:** 14 % of crashes could not be placed in a ZIP; the 3,391 recovered ZIPs are ~94 % accurate.
- **Time:** housing dates back to 2014 and crashes only to 2022; 2026 covers six months.
- **Ecological fallacy:** a crash in a ZIP is not necessarily a crash involving that ZIP's residents.
- **Unit of analysis:** ZIP codes vary greatly in size and are postal routes, not neighbourhoods.

**Impact — who can use this**
- **DOT / Vision Zero:** prioritise street-safety projects in ZIPs that are both top-housing and top-crash (south Bronx 10451–10457, East New York 11207/11208, Brownsville 11212). The most residents with few alternatives to walking and transit live there.
- **HPD / City Planning:** when placing new affordable housing in Queens or southeast Brooklyn, pair it with street-safety review. These areas carry the highest crash burden per unit.
- **Housing advocates and community boards:** use the per-unit and share comparisons (Charts 2 and 8) to argue that Queens is under-served by safety investment relative to its crash burden.

## 10. Prompts used and links
**Dataset links:** [Affordable Housing Production by Building — hg8x-zxpr](https://data.cityofnewyork.us/d/hg8x-zxpr) · [SIRTA — xeqp-qz8h](https://data.cityofnewyork.us/d/xeqp-qz8h)

**AI prompts used in this project** (required by the brief, so list every prompt you used):
| Step | Tool | Prompt |
|---|---|---|
| `<fill in>` | `<e.g. ChatGPT / Claude>` | `<paste the prompts your team used while building the first version>` |
| Review | Claude | "check the project description and the ipyn code and the results and tell me if there is something wrong. look throughly at the requirments and make sure every task is done" |
| Revision | Claude | "okay can you fix all of the problems and send me the new one" |